# Homeownership, Debt, and the American Dream

**Team:** Raghav Dewangan, Kartik Gopalani, Kavyan Patel, Rohan Nigam

**Project Proposal**

**Research Question:**
To what extent does the total amount of debt held by a U.S. citizen correlate with the age at which they first purchase a home?

This notebook will explore, analyze, and model this relationship using publicly available datasets.

## 1. Introduction
- Motivation and summary of proposal
- Importance of analyzing debt and homeownership timing

In the current 2026 economic climate, the American Dream of owning a home has shifted from a young adult's reality to an unbecoming fantasy later into adulthood. While high house prices are often blamed, the internal financial health of the average citizen, specifically their non-housing debt, is a silent barrier.

This project is important because it moves beyond just looking at if people buy homes and instead focuses on when they will be able to. By quantifying the delay in acquiring homeownership caused by credit cards, auto loans, and student debt, we can better understand, explore, and visualize the relationship between personal debt, in whatever form that may be, and wealth milestones; in our case, homeownership.

**Research Question:** To what extent does the total amount of debt held by a U.S. citizen correlate with the age at which they first purchase a home?

- Overview of analysis approach

The ultimate goal is to determine whether there is an association between total personal debt and the timing of homeownership. If such a relationship exists, we aim to measure how strong the association is and estimate its magnitude. This leads into our analysis process. We will begin with Exploratory Data Analysis (EDA), using data visualization techniques such as scatter plots and box plots to identify potential patterns or clusters of individuals who purchase homes earlier versus those who experience longer delays. We will also perform data cleaning to address missing values and outliers that could distort the analysis and lead to unreliable results.

---


## 2. Data Acquisition
- Load Federal Reserve SCF, NLSY97, ZHVI, and any supplemental datasets
- Brief description of each dataset

In [ ]:
import pandas as pd

In [ ]:
nls_duration = pd.read_excel("./data/nls/duration-of-employment-ages-18-to-36.xlsx")
nls_duration

#### NLSY97: Number of jobs held (ages 18–36)
This table captures job switching patterns and employment stability over early adulthood.

In [ ]:
nls_jobs = pd.read_excel('./data/nls/number-of-jobs-held-ages-18-to-36.xlsx')
nls_jobs

#### NLSY97: Partner status (ages 27 and 37)
This table provides household/relationship context that may influence homeownership timing.

In [ ]:
nls_partner_status = pd.read_excel('./data/nls/partner-status-at-ages-27-and-37.xlsx')
nls_partner_status

#### NLSY97: Percent weeks employed/unemployed/out of labor force (by education, ages 18–36)
This table captures labor-force attachment and unemployment exposure over time.

In [ ]:
nls_weeks_employment = pd.read_excel('./data/nls/percent-of-weeks-employment-by-education-ages-18-to-36.xlsx')
nls_weeks_employment

#### NLSY97: Training received (ages 18–36)
This table tracks skill-building/training exposure, which can affect earnings and home-buying readiness.

In [ ]:
nls_training = pd.read_excel('./data/nls/trainings-received-from-ages-18-to-36.xlsx')
nls_training

#### SCFP 2022 (Survey of Consumer Finances Public Data)
This dataset provides household-level financial characteristics (assets, debts, income) used to measure debt burden.

In [ ]:
scfp = pd.read_csv('./data/SCFP2022.csv')
scfp

#### USAUCSFRCONDOSMSAMID (FRED Condo Price Index)
This monthly U.S. condo price series is a market-condition control that helps separate debt effects from housing market effects.

In [ ]:
usa_condo_index = pd.read_csv('./data/USAUCSFRCONDOSMSAMID.csv')
usa_condo_index.head()

In [ ]:
datasets = {
    'nls_duration': nls_duration,
    'nls_jobs': nls_jobs,
    'nls_partner_status': nls_partner_status,
    'nls_weeks_employment': nls_weeks_employment,
    'nls_training': nls_training,
    'scfp': scfp,
    'usa_condo_index': usa_condo_index,
}

for name, d in datasets.items():
    print(f'\n=== {name} ===')
    print('shape:', d.shape)
    print('columns:', d.columns.tolist()[:15])

In [ ]:
from pathlib import Path

nls_paths = sorted(Path('./data/nls').glob('*.xlsx'))
for p in nls_paths:
    print(f'\n=== {p.name} ===')
    xls = pd.ExcelFile(p)
    print('sheets:', xls.sheet_names)
    raw = pd.read_excel(p, sheet_name=xls.sheet_names[0], header=None)
    print(raw.head(8).to_string(index=False, header=False))

In [ ]:
from pathlib import Path

nls_paths = sorted(Path('./data/nls').glob('*.xlsx'))
for p in nls_paths:
    xls = pd.ExcelFile(p)
    raw = pd.read_excel(p, sheet_name=xls.sheet_names[0], header=None, nrows=3)
    print(f"{p.name} | sheet={xls.sheet_names[0]} | A1={raw.iloc[0,0]}")

---


### Using NLS XLSX Tables Correctly (Important)

These NLS files are summary tables, not person-level panel records. Use them for:
- descriptive context,
- trend visualization,
- robustness/triangulation.

Avoid treating them as individual-level observations for causal claims.

### Standardized NLS Loader

This function reads NLS publication-style XLSX tables, detects a likely header row, removes footnote/title artifacts, and returns:
- a cleaned wide table, and
- a tidy long table (`group`, `measure`, `value`).

In [ ]:
import re
from pathlib import Path
import pandas as pd


def _score_header_row(row_values):
    vals = [str(v).strip() for v in row_values if pd.notna(v) and str(v).strip() != '']
    if not vals:
        return -1

    text = ' '.join(vals).lower()
    non_empty = len(vals)

    score = non_empty
    if 'table' in text:
        score -= 3
    if any(k in text for k in ['education', 'age', 'sex', 'race', 'ethnicity', 'hispanic', 'status', 'total']):
        score += 3
    if any('unnamed' in v.lower() for v in vals):
        score -= 1
    return score


def _is_mostly_numeric_header(header_series):
    vals = [str(v).strip() for v in header_series.tolist() if str(v).strip() != '']
    if not vals:
        return True
    numeric_count = 0
    for v in vals:
        try:
            float(v)
            numeric_count += 1
        except Exception:
            pass
    return numeric_count / max(1, len(vals)) >= 0.5


def load_nls_table(path, sheet_name=0, header_row='auto'):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    raw = raw.dropna(axis=0, how='all').dropna(axis=1, how='all').reset_index(drop=True)

    if header_row == 'auto':
        top = min(20, len(raw))
        scored = [(_score_header_row(raw.iloc[i].tolist()), i) for i in range(top)]
        header_idx = max(scored, key=lambda x: x[0])[1]
    else:
        header_idx = int(header_row)

    header = raw.iloc[header_idx].astype(str).str.strip().replace({'nan': ''})

    if _is_mostly_numeric_header(header) and header_idx > 0:
        candidate = raw.iloc[header_idx - 1].astype(str).str.strip().replace({'nan': ''})
        if not _is_mostly_numeric_header(candidate):
            header = candidate
            header_idx = header_idx - 1

    header = [h if h != '' else f'col_{i}' for i, h in enumerate(header)]

    data = raw.iloc[header_idx + 1:].copy()
    data.columns = header
    data = data.dropna(axis=0, how='all').reset_index(drop=True)

    first_col = data.columns[0]
    data = data.rename(columns={first_col: 'group'})

    def clean_group(x):
        s = str(x)
        s = re.sub(r'\s+', ' ', s).strip()
        s = re.sub(r'[\.·…]+$', '', s).strip()
        return s

    data['group'] = data['group'].map(clean_group)

    long_df = data.melt(id_vars=['group'], var_name='measure', value_name='value')
    long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')
    long_df = long_df.dropna(subset=['value']).reset_index(drop=True)

    return data, long_df


nls_file_map = {
    'duration': './data/nls/duration-of-employment-ages-18-to-36.xlsx',
    'jobs': './data/nls/number-of-jobs-held-ages-18-to-36.xlsx',
    'partner_status': './data/nls/partner-status-at-ages-27-and-37.xlsx',
    'weeks_employment': './data/nls/percent-of-weeks-employment-by-education-ages-18-to-36.xlsx',
    'training': './data/nls/trainings-received-from-ages-18-to-36.xlsx',
}

nls_tables_wide = {}
nls_tables_long = {}

for key, filepath in nls_file_map.items():
    wide_df, long_df = load_nls_table(filepath)
    nls_tables_wide[key] = wide_df
    nls_tables_long[key] = long_df
    print(f"{key:16s} wide={wide_df.shape} | long={long_df.shape}")

nls_tables_long['jobs'].head()

### NLS Table Display and Visualization (EDA)

These cells show each cleaned NLS table and create comparable plots across all NLS datasets.

**Note on `Total 2`:** In these NLS tables, `Total` is the overall estimate across all individuals/groups in the table (for the stated age range and period). The trailing `2` is a table footnote marker from the original BLS/NLS spreadsheet, not part of the metric name itself.

In [ ]:
from IPython.display import display, Markdown

for key, table in nls_tables_wide.items():
    display(Markdown(f"#### {key.replace('_', ' ').title()} (Wide)"))
    display(table)

for key, table in nls_tables_long.items():
    display(Markdown(f"#### {key.replace('_', ' ').title()} (Long/Tidy)"))
    display(table.head(20))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
axes = axes.flatten()

for i, (key, long_df) in enumerate(nls_tables_long.items()):
    ax = axes[i]

    measures = [m for m in long_df['measure'].dropna().unique().tolist() if str(m).strip() != '']
    chosen_measure = 'Total 2' if 'Total 2' in measures else measures[0]

    plot_df = long_df[long_df['measure'] == chosen_measure].copy()
    plot_df = plot_df[~plot_df['group'].str.contains('total', case=False, na=False)]
    plot_df = plot_df.sort_values('value', ascending=False).head(12)

    ax.barh(plot_df['group'], plot_df['value'])
    ax.invert_yaxis()
    ax.set_title(f"{key.replace('_', ' ').title()} ({chosen_measure})")
    ax.set_xlabel('Value')
    ax.set_ylabel('Group')

if len(nls_tables_long) < len(axes):
    for j in range(len(nls_tables_long), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def clean_nls_summary_table(path):
    raw = pd.read_excel(path, header=None)
    raw = raw.dropna(axis=0, how='all').dropna(axis=1, how='all').reset_index(drop=True)

    candidate_rows = []
    for i in range(min(12, len(raw))):
        row = raw.iloc[i]
        non_null = row.notna().sum()
        text = ' '.join(str(v) for v in row.dropna().astype(str).tolist()).lower()
        score = non_null
        if 'table' in text:
            score -= 3
        if 'age' in text or 'education' in text or 'sex' in text:
            score += 2
        candidate_rows.append((score, i))

    header_idx = sorted(candidate_rows, reverse=True)[0][1]
    header = raw.iloc[header_idx].astype(str).str.strip().replace({'nan': ''})

    df = raw.iloc[header_idx + 1:].copy()
    df.columns = header
    df = df.loc[:, df.columns != '']
    df = df.dropna(axis=0, how='all').reset_index(drop=True)

    first_col = df.columns[0]
    df = df.rename(columns={first_col: 'group'})

    long_df = df.melt(id_vars=['group'], var_name='metric_or_age', value_name='value')
    long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')
    long_df = long_df.dropna(subset=['value']).reset_index(drop=True)
    return df, long_df

nls_files = {
    'duration': './data/nls/duration-of-employment-ages-18-to-36.xlsx',
    'jobs': './data/nls/number-of-jobs-held-ages-18-to-36.xlsx',
    'partner_status': './data/nls/partner-status-at-ages-27-and-37.xlsx',
    'weeks_employment': './data/nls/percent-of-weeks-employment-by-education-ages-18-to-36.xlsx',
    'training': './data/nls/trainings-received-from-ages-18-to-36.xlsx',
}

nls_wide = {}
nls_long = {}
for name, path in nls_files.items():
    wide, long = clean_nls_summary_table(path)
    nls_wide[name] = wide
    nls_long[name] = long
    print(f"{name}: wide={wide.shape}, long={long.shape}")

nls_long['jobs'].head()

In [ ]:
import hashlib
from pathlib import Path

for p in sorted(Path('./data/nls').glob('*.xlsx')):
    file_hash = hashlib.md5(p.read_bytes()).hexdigest()
    print(p.name, file_hash)

In [ ]:
from pathlib import Path

for p in sorted(Path('./data/nls').glob('*.xlsx')):
    xls = pd.ExcelFile(p)
    print(p.name, '->', xls.sheet_names)

## 3. Data Exploration & Cleaning
- EDA: summary statistics, missing values, outlier handling
- Visualizations: scatter plots, box plots, debt binning

---


## 4. Data Merging
- Combine datasets for unified analysis

---


## 5. Modeling
- Multiple Linear Regression (age vs. debt/income)
- Variable selection
- Optionally: Logistic Regression (homeownership yes/no)
- Cross-validation

---


### Hypothesis Test: Debt and Age at First Home Purchase

**Null Hypothesis ($H_0$):** There is no association between total debt and the age at which a U.S. citizen first purchases a home.

**Alternative Hypothesis ($H_1$):** There is a significant association between total debt and the age at which a U.S. citizen first purchases a home.

We will use a t-test on the slope of a linear regression model to test this hypothesis. If the p-value is below a chosen significance level (e.g., 0.05), we reject the null hypothesis.

## 6. Results & Discussion
- Key findings, metrics (e.g., debt-to-income tipping point)
- Interpretation of results

---


## 7. Conclusion
- Summary, limitations, future directions

---


## 8. References
- Data sources, external code/libraries